## Домашнее задание 2. Реализация обучения диффузионной модели

###Цель домашнего занятия
Научиться реализовывать обучение диффузии под датасет и решить задачу Image Inpainting

### [6 баллов] Дообучение диффузионной модели.

В этом задании вам предстоит обучить диффузионную модель для решения задачи text-to-image. В предыдущем задании вы уже обучали диффузионную модель, но она была unconditional, то есть генерировала без каких-либо условий. В этот раз вам предлагается обучить свою text-to-image модель на датасете https://huggingface.co/datasets/Norod78/cartoon-blip-captions или на другом понравившемся датасете

---

В этой части задания вам рекомендуется взять код с семинара 4 и адаптировать его под обучение на новом датасете, с учетом того, что в нем содержаться картинки

**Ожидаемый результат.** В качестве результатов модели, от вас требуется предоставить визуализации генераций, полученных от вашей модели и код обучения модели, а также визуальное сравнение качества моделей до и после дообучения. Если не получится дообучить, то можно провести сравнение уже обученных адаптеров (3 балла).

In [ ]:
#import matplotlib.pyplot as plt
from datasets import load_dataset

In [ ]:
# !uv run huggingface-cli download --repo-type dataset "Norod78/cartoon-blip-captions"

Fetching 3 files:   0%|                                   | 0/3 [00:00<?, ?it/s]Downloading 'data/train-00000-of-00001-dfb0d9df7ebab67e.parquet' to '/home/lev/.cache/huggingface/hub/datasets--Norod78--cartoon-blip-captions/blobs/720a75399d52fff9657bea668b4b759a1a6dc87aa82f8f35a574d91b2b7e3bc8.incomplete'

README.md: 100%|███████████████████████████████| 536/536 [00:00<00:00, 2.16MB/s]
Download complete. Moving file to /home/lev/.cache/huggingface/hub/datasets--Norod78--cartoon-blip-captions/blobs/dea2d87be60204f3a64f4c743ab0787aa2eef673

(…)-00000-of-00001-dfb0d9df7ebab67e.parquet:   0%|   | 0.00/190M [00:00<?, ?B/s]Downloading '.gitattributes' to '/home/lev/.cache/huggingface/hub/datasets--Norod78--cartoon-blip-captions/blobs/7370acfb01ec70391e9e2389f65b95a5d5c87334.incomplete'

(…)-00000-of-00001-dfb0d9df7ebab67e.parquet:  11%| | 21.0M/190M [00:00<00:01, 15

.gitattributes: 2.22kB [00:00, 5.09MB/s]A
Download complete. Moving file to /home/lev/.cache/huggingface/hub/datasets--Norod78-

In [6]:
# https://huggingface.co/datasets/Norod78/cartoon-blip-captions
# config.dataset_name = "Norod78/cartoon-blip-captions"
# name = "https://huggingface.co/datasets/Norod78/cartoon-blip-captions"
name = "Norod78/cartoon-blip-captions"
dataset = load_dataset(name, split="train")
print(dataset)

Generating train split:   0%|          | 0/3141 [00:00<?, ? examples/s]

Dataset({
    features: ['image', 'text'],
    num_rows: 3141
})


In [7]:
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for i, image in enumerate(dataset[:4]["image"]):
    axs[i].imshow(image)
    axs[i].set_axis_off()
fig.show()

for i, text in enumerate(dataset[:4]["text"]):
    print(f"{i}: {text}")

NameError: name 'plt' is not defined

#### Установка kohya-ss/sd-scripts

1. Клонируем  kohya-ss/sd-scripts

```shell
lev@mild-fire-smells-fin-01:~/project/ml-modern/homework/hw02$ git clone https://github.com/kohya-ss/sd-scripts.git
Cloning into 'sd-scripts'...
remote: Enumerating objects: 10774, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 10774 (delta 28), reused 18 (delta 18), pack-reused 10725 (from 2)
Receiving objects: 100% (10774/10774), 12.54 MiB | 13.15 MiB/s, done.
Resolving deltas: 100% (7710/7710), done.
```

2. Установка venv

```shell
lev@mild-fire-smells-fin-01:~/project/ml-modern/homework/hw02$ uv init --python 3.10 --no-readme --no-package --no-description
Initialized project `hw02`
lev@mild-fire-smells-fin-01:~/project/ml-modern/homework/hw02$ uv venv --python 3.10 /home/lev/.cache/uv/virtualenvs/ml-modern-hw02
Using CPython 3.10.19
Creating virtual environment at: /home/lev/.cache/uv/virtualenvs/ml-modern-hw02
Activate with: source /home/lev/.cache/uv/virtualenvs/ml-modern-hw02/bin/activate
lev@mild-fire-smells-fin-01:~/project/ml-modern/homework/hw02$ ln -s /home/lev/.cache/uv/virtualenvs/ml-modern-hw02 .venv
lev@mild-fire-smells-fin-01:~/project/ml-modern/homework/hw02$ source /home/lev/.cache/uv/virtualenvs/ml-modern-hw02/bin/activate
```

3. Руками добавил requirements.txt в pyproject.toml

```shell
(ml-modern-hw02) lev@mild-fire-smells-fin-01:~/project/ml-modern/homework/hw02$ uv lock
Resolved 127 packages in 1.54s
(ml-modern-hw02) lev@mild-fire-smells-fin-01:~/project/ml-modern/homework/hw02$ uv sync
Resolved 127 packages in 2ms
Prepared 60 packages in 16.08s
Installed 118 packages in 4.28s
```

4. Установка library из kohya-ss/sd-scripts

Дописываем в pyproject.toml:
```toml
[tool.uv.sources]
library = { path = "/home/lev/mipt.project/ml-modern/homework/hw02/sd-scripts", editable = true }
```

```shell
(ml-modern-hw02) lev@mild-fire-smells-fin-01:~/project/ml-modern/homework/hw02$ uv lock
Resolved 124 packages in 518ms
Added library v0.0.0
(ml-modern-hw02) lev@mild-fire-smells-fin-01:~/project/ml-modern/homework/hw02$ uv sync
Resolved 124 packages in 2ms
Audited 119 packages in 3ms
```

- Готовые LoRA слои можно загрузить с сайта [Civitai](https://civitai.com/)**
- Diffusers предоставляет [скрипт](https://github.com/huggingface/diffusers/blob/main/examples/text_to_image/train_text_to_image_lora.py) тонкой настройки LoRA, который может работать всего в 11 ГБ оперативной памяти GPU, не прибегая к таким трюкам, как 8-битные оптимизаторы. Вот как его можно использовать для тонкой настройки модели на наборе данных Lambda Labs Pokémon
- внимание тот факт, что learning rate составляет 1e-4, что значительно больше, чем обычный learning rate при finetuning (обычно она составляет ~1e-6). 
    - Это [дашборд W&B](https://wandb.ai/pcuenq/text2image-fine-tune/runs/b4k1w0tn?workspace=user-pcuenq), который занял около 5 часов на GPU 2080 Ti (11 ГБ оперативной памяти) 
    - и пример [демо](https://huggingface.co/spaces/pcuenq/lora-pokemon)

Workflow on seminar 4
- установка kohya trainer
- open file explorer
- установка предобученной модели
- установка пользовательской модели (пользовательских параметров)
- установка доступного VAE Variable AutoEncoder (опционально)

Получение данных
- ImageScraper (optional) - скачать покемонов
- Annotation
    - BLIP Captioning
    - Waifu Diffusion Tagger - classifies and adds tags in Stable Diffusion format
    - Metadata file
    - Bucketing and caching

Training
- LoRA and Optimizer Config
- prompt
- accelerate_conf, train_conf
- accelerate launch

Testing
- Check weights
- load_file, load_weights
- generate image


#### 1.1 Установка зависимостей

#### 2.1 Установка доступной модели

#### 2.3 Установка доступного VAE (опционально)

#### 4.2.2 Waifu Diffusion 1.4 Tagger V2

#### 4.3 Создаем файл метаданных

In [ ]:
#Визулизации

#### Выводы и т. п.

Попытка 1.

Пробовал запустить ноутбук из семинара 4. Видимо он запускался в старом колабе с питоном 3.10.
Сейчас у колаба питон 3.12 и 3.11. 
Пытался порешить проблемы с зависимостями. Использовал uv, локальный 
Linux Ubuntu 24.04с 1 x RTX 3080 10GB.
Так и не получилось. Сам использованный kohya-trainer deprecated и в архиве.

Попытка 2.

Пробую актульную версию [kohya-ss/sd-scripts](https://github.com/kohya-ss/sd-scripts) на Linux Ubuntu 24.04 с 1 x NVIDIA A100-SXM4-80GB.




### [4 балла] Image Inpainting
В первой части задания (2 балла) вам предстоит обучить диффузионную модель решать задачу image inpainting. Для этого вам необходимо модифицировать код из семинара 2 следующий образом:
- Реализовать функцию генерации маски, можно использовать приведенную ниже или реализовать свою.
- Добавить в UNet новых 4 канала, куда подавать замаскированное изображение и маску.
- Запустить обучение.

---

**Ожидаемый результат.** В качестве результатов модели, от вас требуется предоставить визуализации генераций, полученных от вашей модели и код обучения модели.

In [ ]:
import numpy as np

class RectangleGenerator:
    """
    Generates for each object a mask where unobserved region is
    a rectangle which square divided by the image square is in
    interval [min_rect_rel_square, max_rect_rel_square].
    """
    def __init__(self, min_rect_rel_square=0.3, max_rect_rel_square=1):
        self.min_rect_rel_square = min_rect_rel_square
        self.max_rect_rel_square = max_rect_rel_square

    def gen_coordinates(self, width, height):
        x1, x2 = np.random.randint(0, width, 2)
        y1, y2 = np.random.randint(0, height, 2)
        x1, x2 = min(x1, x2), max(x1, x2)
        y1, y2 = min(y1, y2), max(y1, y2)
        return int(x1), int(y1), int(x2), int(y2)

    def __call__(self, batch):
        batch_size, num_channels, width, height = batch.shape
        mask = torch.zeros_like(batch)
        for i in range(batch_size):
            x1, y1, x2, y2 = self.gen_coordinates(width, height)
            sqr = width * height
            while not (self.min_rect_rel_square * sqr <=
                       (x2 - x1 + 1) * (y2 - y1 + 1) <=
                       self.max_rect_rel_square * sqr):
                x1, y1, x2, y2 = self.gen_coordinates(width, height)
            mask[i, :, x1: x2 + 1, y1: y2 + 1] = 1
        return mask

Во второй части задания (2 балла) вам предстоит провизуализировать результаты полученной модели inpainting и модели SD через [демо](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/in_painting_with_stable_diffusion_using_diffusers.ipynb#scrollTo=R596bpT2ynqV). Пример демо реализован ниже. Если у вас не получится реализовать модель из первого задания, то вы можете использовать модель SD и получите один балл

**Ожидаемый результат.** От вас ожидается анализ кейсов, когда модель отрабатывает хорошо (например, определенный тип масок), а когда плохо.

In [ ]:
# Визуализации